In [ ]:
#处理数据
#选取数据
r_action_item_ids = product_data['item_id'].tolist()
filtered_chunk = chunk[chunk['item_id'].isin(r_action_item_ids)]
from pyspark.sql.functions import regexp_replace
# 删除rater_uid字段中的'u'
action = action.withColumn('user_id', regexp_replace('user_id', 'u', ''))
from pyspark.sql.functions import to_timestamp, date_format
# 将'r_vtime'字段转换为指定格式的日期字符串 'YYYY-MM-DD'
action = action.withColumn('r_vtime', date_format(to_timestamp('vtime'), 'yyyy-MM-dd'))
# 对item_id列进行升序排序
action = action.orderBy(asc('item_id'))
# 使用groupby和agg进行分组并统计数量
result = df.groupby(['item_id', 'action']).agg(count=('user_id', 'count')).reset_index()
# 使用pivot函数将action字段拆分为四个字段
df_pivot = df.pivot(index='item_id', columns='action', values='count').reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)
# 填充缺失值（NaN）并重命名列名
df_pivot = df_pivot.fillna(0)
df_pivot = df_pivot.astype({'alipay': int, 'cart': int, 'click': int, 'collect': int})

# 目录
1. [用户行为数据处理](#section1)
 * 1.1 [转化数据存储类型](#section1_1)
 * 1.2 [合并](#section1_2)
 * 1.3 [规范化数据](#section1_3)
 * 1.4 [统计用户行为](#section1_4)
2. [用户行为创建ALS模型](#section2)
 * 2.1 [创建Spark会话](#section2_1)
 * 2.2 [加载数据](#section2_2)
 * 2.3 [根据用户对每件商品偏好打分训练ALS模型](#section2_3)
 * 2.4 [评估](#section2_4)
3. [优质用户](#section3)

# 1. 用户行为数据处理 <a class="anchor" id="section1"></a>

>有三个数据集，且每个数据集数据量较大，考虑一个一个的处理出来之后再合并所有用户行为数据

## 1.1 转化数据存储类型 <a class="anchor" id="section1_1"></a>

In [5]:
import csv

output_file = open('D:\\Datas\\datas\\action.csv', 'a', newline='', encoding='utf-8')
csv_writer = csv.writer(output_file)

# 设置字段名
field_names = ['item_id', 'user_id', 'action','vtime']
csv_writer.writerow(field_names)

file_paths = ['D:\\Datas\\tianchi_2014002_rec_tmall_log.parta\\tianchi_2014002_rec_tmall_log_parta.txt',
              'D:\\Datas\\tianchi_2014002_rec_tmall_log.partb\\tianchi_2014002_rec_tmall_log_partb.txt',
              'D:\\Datas\\tianchi_2014002_rec_tmall_log.partc\\tianchi_2014002_rec_tmall_log_partc.txt']

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            # 假设每行数据格式为 "item_idother_columns"
            data_fields = line.strip().split('')
            csv_writer.writerow(data_fields)

output_file.close()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Datas\\tianchi_2014002_rec_tmall_log.parta\\tianchi_2014002_rec_tmall_log_parta.txt'

In [7]:
import pandas as pd

# 定义分块大小
chunk_size = 5000000

chunks = []
for chunk in pd.read_csv('D:\\Datas\\datas\\action.csv', chunksize=chunk_size):
    # 处理每个chunk的数据
    # 根据商品数据的item_id筛选行为数据
    product_data = pd.read_csv('D:\\Datas\\datas\\product_1.csv')  # 读取商品数据
    r_action_item_ids = product_data['item_id'].tolist()
    filtered_chunk = chunk[chunk['item_id'].isin(r_action_item_ids)]
    
    # 将筛选后的数据保存为单独的CSV文件
    filtered_chunk.to_csv(f'D:\\Datas\\datas\\filtered_chunk_{len(chunks)}.csv', index=False)
    
    chunks.append(filtered_chunk)

# 合并所有筛选后的数据块
filtered_data = pd.concat(chunks, ignore_index=True)

# 查看筛选后的用户行为数据前10行
print(filtered_data.head(10))


F:\Anaconda\procedure\lib\site-packages\IPython\core\interactiveshell.py:3364: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


   item_id   user_id action                vtime
0  6289870  u3427707  click  2013-08-30 09:27:15
1  4120848  u3427707  click  2013-05-22 16:54:34
2  4120848  u3427707  click  2013-05-24 12:27:31
3  4690166  u3427707  click  2013-05-22 16:55:10
4  4690166  u3427707  click  2013-05-24 12:27:39
5  4690166  u3427707  click  2013-05-22 16:54:34
6  6301705  u3427707  click  2013-07-29 16:43:02
7  6301705  u3427707  click  2013-07-29 16:54:39
8  6025858  u3427707  click  2013-06-04 12:35:32
9  1528104  u5853382  click  2013-09-16 13:10:44


In [8]:
filtered_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93703874 entries, 0 to 93703873
Data columns (total 4 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   item_id  object
 1   user_id  object
 2   action   object
 3   vtime    object
dtypes: object(4)
memory usage: 2.8+ GB


## 1.2 合并 <a class="anchor" id="section1_2"></a>

In [9]:
#filtered_data是合并后的数据
filtered_data.to_csv('D:\\Datas\\datas\\filtered_data.csv', index=False)

## 1.3 规范化数据 <a class="anchor" id="section1_3"></a>

### 1.3.1 规范用户ID

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import asc

# spark配置信息
from pyspark import SparkConf

SPARK_APP_NAME = "Action_1"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "10g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)

conf.setAll(config)

# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

# 读取筛选后的数据为DataFrame
action = spark.read.csv("hdfs://localhost:9000//E_commerce_platform/action/filtered_data.csv", header=True, inferSchema=True)
action.show()

+-------+---------+------+-------------------+
|item_id|  user_id|action|              vtime|
+-------+---------+------+-------------------+
|6289870| u3427707| click|2013-08-30 09:27:15|
|4120848| u3427707| click|2013-05-22 16:54:34|
|4120848| u3427707| click|2013-05-24 12:27:31|
|4690166| u3427707| click|2013-05-22 16:55:10|
|4690166| u3427707| click|2013-05-24 12:27:39|
|4690166| u3427707| click|2013-05-22 16:54:34|
|6301705| u3427707| click|2013-07-29 16:43:02|
|6301705| u3427707| click|2013-07-29 16:54:39|
|6025858| u3427707| click|2013-06-04 12:35:32|
|1528104| u5853382| click|2013-09-16 13:10:44|
|4914919| u5853382| click|2013-09-14 22:33:07|
|6183967| u5853382| click|2013-09-16 15:46:14|
|3418158| u5853382| click|2013-09-07 13:26:36|
|3294946| u5853382| click|2013-07-13 15:19:39|
|7328508| u5853382| click|2013-09-07 15:49:34|
|4783514|u11408420| click|2013-06-18 23:00:07|
|5166278|u11408420| click|2013-06-02 22:54:00|
|1892082|u11408420| click|2013-08-24 00:40:59|
|4227691|u114

In [2]:
from pyspark.sql.functions import regexp_replace
# 删除rater_uid字段中的'u'
action = action.withColumn('user_id', regexp_replace('user_id', 'u', ''))
# 显示处理后的DataFrame前5行
action.show(5)

+-------+-------+------+-------------------+
|item_id|user_id|action|              vtime|
+-------+-------+------+-------------------+
|6289870|3427707| click|2013-08-30 09:27:15|
|4120848|3427707| click|2013-05-22 16:54:34|
|4120848|3427707| click|2013-05-24 12:27:31|
|4690166|3427707| click|2013-05-22 16:55:10|
|4690166|3427707| click|2013-05-24 12:27:39|
+-------+-------+------+-------------------+
only showing top 5 rows



### 1.3.2 规范时间戳

In [3]:
from pyspark.sql.functions import to_timestamp, date_format
# 将'r_vtime'字段转换为指定格式的日期字符串 'YYYY-MM-DD'
action = action.withColumn('r_vtime', date_format(to_timestamp('vtime'), 'yyyy-MM-dd'))
# 显示处理后的DataFrame前5行
action.show(5)

+-------+-------+------+-------------------+----------+
|item_id|user_id|action|              vtime|   r_vtime|
+-------+-------+------+-------------------+----------+
|6289870|3427707| click|2013-08-30 09:27:15|2013-08-30|
|4120848|3427707| click|2013-05-22 16:54:34|2013-05-22|
|4120848|3427707| click|2013-05-24 12:27:31|2013-05-24|
|4690166|3427707| click|2013-05-22 16:55:10|2013-05-22|
|4690166|3427707| click|2013-05-24 12:27:39|2013-05-24|
+-------+-------+------+-------------------+----------+
only showing top 5 rows



### 1.3.3 规范item_id

In [4]:
# 对item_id列进行升序排序
action = action.orderBy(asc('item_id'))
# 显示排序后的结果
action.show(5)

+-------+--------+------+-------------------+----------+
|item_id| user_id|action|              vtime|   r_vtime|
+-------+--------+------+-------------------+----------+
|    163| 7615221| click|2013-06-26 11:23:44|2013-06-26|
|    163| 2243121| click|2013-07-01 18:10:19|2013-07-01|
|    163| 9767215| click|2013-07-01 22:58:29|2013-07-01|
|    163|12500579| click|2013-06-26 09:06:41|2013-06-26|
|    163| 5035937| click|2013-07-05 16:13:17|2013-07-05|
+-------+--------+------+-------------------+----------+
only showing top 5 rows



### 1.3.4 保存

In [15]:
# 将处理后的数据保存为CSV文件 数据太大导致存储错误
# action.write.csv("hdfs://localhost:9000/E_commerce_platform/action/action", header=True)
# 将处理后的数据保存为单个CSV文件到本地
# action.coalesce(1).write.csv("file:///D:/Datas/datas/action_1.csv", header=True)
action.show(5)

+-------+--------+------+-------------------+----------+
|item_id| user_id|action|              vtime|   r_vtime|
+-------+--------+------+-------------------+----------+
|    163| 2868413| click|2013-06-23 10:22:14|2013-06-23|
|    163| 2243121| click|2013-07-01 18:10:19|2013-07-01|
|    163|11904148| click|2013-06-29 16:51:38|2013-06-29|
|    163|12500579| click|2013-06-26 09:06:41|2013-06-26|
|    163| 5035937| click|2013-07-05 16:13:17|2013-07-05|
+-------+--------+------+-------------------+----------+
only showing top 5 rows



In [ ]:
import pandas as pd

# 将Spark DataFrame转换为Pandas DataFrame
limited_action_df = action.limit(250000).toPandas()
# 将Pandas DataFrame保存为CSV文件
limited_action_df.to_csv("D:/Datas/datas/action_data_1.csv", index=False)

对数据进行操作后十分的吃力，电脑额内存完全无法支持海量的数据的保存，因此只能考虑保存部分数据，以此实现正常的分析展示。

In [6]:
# 停止SparkSession
spark.stop()

## 1.4 统计用户行为  <a class="anchor" id="section1_4"></a>

### 1.4.1 统计购买量、点击量、收藏量、加入购物车量

In [8]:
import pandas as pd

# 读取CSV文件为Pandas DataFrame
action = pd.read_csv("D:/Datas/datas/action_data_1.csv")
action.head(10)

,item_id,user_id,action,vtime,r_vtime
0,163,6714091,click,2013-07-09 13:06:49,2013-07-09
1,163,7900663,collect,2013-06-27 15:09:55,2013-06-27
2,163,3065348,click,2013-06-29 14:35:26,2013-06-29
3,163,14494972,click,2013-06-27 14:54:54,2013-06-27
4,163,11019638,click,2013-07-12 17:35:54,2013-07-12
5,163,5035937,click,2013-07-05 16:13:17,2013-07-05
6,163,175857,click,2013-06-23 15:06:30,2013-06-23
7,163,11278011,click,2013-06-24 11:04:54,2013-06-24
8,163,13415493,click,2013-07-03 08:32:29,2013-07-03
9,163,10296125,click,2013-07-08 13:26:20,2013-07-08


In [10]:
df = pd.DataFrame(action)

# 使用groupby和agg进行分组并统计数量
result = df.groupby(['item_id', 'action']).agg(count=('user_id', 'count')).reset_index()
result.head(10)

,item_id,action,count
0,163,alipay,1
1,163,cart,4
2,163,click,113
3,163,collect,1
4,183,alipay,6
5,183,cart,17
6,183,click,666
7,183,collect,25
8,320,alipay,6
9,320,cart,10


### 1.4.2  拆分action列

In [11]:
#方便查找
import pandas as pd

df = pd.DataFrame(result)

# 使用pivot函数将action字段拆分为四个字段
df_pivot = df.pivot(index='item_id', columns='action', values='count').reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)

# 填充缺失值（NaN）并重命名列名
df_pivot = df_pivot.fillna(0)
df_pivot = df_pivot.astype({'alipay': int, 'cart': int, 'click': int, 'collect': int})
df_pivot.head()

,item_id,alipay,cart,click,collect
0,163,1,4,113,1
1,183,6,17,666,25
2,320,6,10,85,0
3,331,2,10,155,1
4,343,4,1,246,4


### 1.4.3 保存

In [12]:
#保存方便渲染页面
df_pivot.to_csv('D:\\Datas\\datas\\item_action_count.csv', index=False)

# 2. 用户行为创建ALS模型  <a class="anchor" id="section2"></a>

In [63]:
import findspark
findspark.init()

## 2.1 创建Spark会话 <a class="anchor" id="section2_1"></a>

In [65]:
# spark配置信息
from pyspark import SparkConf
from pyspark.sql import SparkSession

SPARK_APP_NAME = "ActionRecommendationSystem"

conf = SparkConf()    # 创建spark config对象
config = (
    ("spark.app.name", SPARK_APP_NAME),    # 设置启动的spark的app名称，没有提供，将随机产生一个名称
    ("spark.executor.memory", "8g"),    # 设置该app启动时占用的内存用量，默认1g
    ("spark.executor.cores", "4"),    # 设置spark executor使用的CPU核心数
)


conf.setAll(config)

# 利用config对象，创建spark session
spark = SparkSession.builder.config(conf=conf).getOrCreate()

## 2.2 加载数据 <a class="anchor" id="section2_2"></a>

In [66]:
# 从hdfs加载CSV文件为DataFrame
df = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/action/action_data_1.csv", header=True)
df.show()    # 查看dataframe，默认显示前20条
# 大致查看一下数据类型
df.printSchema()    # 打印当前dataframe的结构

+-------+--------+-------+-------------------+----------+
|item_id| user_id| action|              vtime|   r_vtime|
+-------+--------+-------+-------------------+----------+
|    163| 6714091|  click|2013-07-09 13:06:49|2013-07-09|
|    163| 7900663|collect|2013-06-27 15:09:55|2013-06-27|
|    163| 3065348|  click|2013-06-29 14:35:26|2013-06-29|
|    163|14494972|  click|2013-06-27 14:54:54|2013-06-27|
|    163|11019638|  click|2013-07-12 17:35:54|2013-07-12|
|    163| 5035937|  click|2013-07-05 16:13:17|2013-07-05|
|    163|  175857|  click|2013-06-23 15:06:30|2013-06-23|
|    163|11278011|  click|2013-06-24 11:04:54|2013-06-24|
|    163|13415493|  click|2013-07-03 08:32:29|2013-07-03|
|    163|10296125|  click|2013-07-08 13:26:20|2013-07-08|
|    163| 1316194|  click|2013-07-08 13:04:56|2013-07-08|
|    163| 1932168|  click|2013-06-28 15:25:27|2013-06-28|
|    163|  854467|  click|2013-07-03 19:05:57|2013-07-03|
|    163|10362637|  click|2013-07-03 22:10:14|2013-07-03|
|    163|10409

### 2.2.1 设置字段 

In [67]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType
from pyspark.sql.functions import unix_timestamp
from pyspark.sql.functions import from_unixtime
# 构建结构对象
schema = StructType([
    StructField("item_id", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("action", StringType()),
    StructField("vtime", StringType()),
    StructField("r_vtime", StringType()),
])
# 从hdfs加载数据为dataframe，并设置结构
action_df = spark.read.csv("hdfs://localhost:9000/E_commerce_platform/action/action_data_1.csv", header=True, schema=schema)
action_df.show()
# review_df.count() 
action_df.printSchema() 

+-------+--------+-------+-------------------+----------+
|item_id| user_id| action|              vtime|   r_vtime|
+-------+--------+-------+-------------------+----------+
|    163| 6714091|  click|2013-07-09 13:06:49|2013-07-09|
|    163| 7900663|collect|2013-06-27 15:09:55|2013-06-27|
|    163| 3065348|  click|2013-06-29 14:35:26|2013-06-29|
|    163|14494972|  click|2013-06-27 14:54:54|2013-06-27|
|    163|11019638|  click|2013-07-12 17:35:54|2013-07-12|
|    163| 5035937|  click|2013-07-05 16:13:17|2013-07-05|
|    163|  175857|  click|2013-06-23 15:06:30|2013-06-23|
|    163|11278011|  click|2013-06-24 11:04:54|2013-06-24|
|    163|13415493|  click|2013-07-03 08:32:29|2013-07-03|
|    163|10296125|  click|2013-07-08 13:26:20|2013-07-08|
|    163| 1316194|  click|2013-07-08 13:04:56|2013-07-08|
|    163| 1932168|  click|2013-06-28 15:25:27|2013-06-28|
|    163|  854467|  click|2013-07-03 19:05:57|2013-07-03|
|    163|10362637|  click|2013-07-03 22:10:14|2013-07-03|
|    163|10409

### 2.2.2 分析字段

In [24]:
print("查看item_id的数据情况：", action_df.groupBy("item_id").count().count())
# 340个商品
#review_df.groupBy("item_id").count()  返回的是一个dataframe，这里的count计算的是每一个分组的个数，但当前还没有进行计算
# 当调用df.count()时才开始进行计算，这里的count计算的是dataframe的条目数，也就是共有多少个分组

查看item_id的数据情况： 340


In [25]:
print("查看user_id的数据情况：", action_df.groupBy("user_id").count().count())
#164674个用户

查看user_id的数据情况： 164674


In [26]:
print("查看action的数据情况：", action_df.groupBy("action").count().collect())    # collect会把计算结果全部加载到内存，谨慎使用
# 只有4种类型数据

查看action的数据情况： [Row(action='alipay', count=1651), Row(action='cart', count=5807), Row(action='click', count=235575), Row(action='collect', count=6967)]


## 2.3 根据用户对每件商品偏好打分训练ALS模型  <a class="anchor" id="section2_3"></a>

In [68]:
# 统计每个用户对每一种商品的alipay、cart、click、collect数量
cate_count_df = action_df.groupBy(action_df.item_id, action_df.user_id).pivot("action",["alipay","cart","click","collect"]).count()
cate_count_df.printSchema()    # 此时还没有开始计算

root
 |-- item_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- alipay: long (nullable = true)
 |-- cart: long (nullable = true)
 |-- click: long (nullable = true)
 |-- collect: long (nullable = true)



In [69]:
def process_row(r):
    # 处理每一行数据：r表示row对象
    # 偏好评分规则：
    # 如果行为次数小于等于20次，则计算得分为行为次数乘以对应的权重（0.2、0.4、0.6、1.0）
    # 如果行为次数大于20次，则得分固定为4、8、12、20
    
    # 注意这里要全部设为浮点数，spark运算时对类型比较敏感，要保持数据类型都一致
    click_count = r.click if r.click else 0.0
    collect_count = r.collect if r.collect else 0.0
    cart_count = r.cart if r.cart else 0.0
    alipay_count = r.alipay if r.alipay else 0.0

    click_score = 0.2*click_count if click_count<=20 else 4.0
    collect_score = 0.4*collect_count if collect_count<=20 else 8.0
    cart_score = 0.6*cart_count if cart_count<=20 else 12.0
    alipay_count = 1.0*alipay_count if alipay_count<=20 else 20.0

    rating = click_score + collect_score + cart_score + alipay_count
    # 返回用户ID、商品ID、用户对分类的偏好打分
    return r.user_id, r.item_id, rating

### 2.3.1 返回一个PythonRDD类型

In [70]:
# 返回一个PythonRDD类型，此时还没开始计算
cate_count_df.rdd.map(process_row).toDF(["user_id", "item_id", "rating"])

DataFrame[user_id: bigint, item_id: bigint, rating: double]

### 2.3.2 用户对商品的打分数据

In [72]:
# 用户对商品的打分数据
# map返回的结果是rdd类型，需要调用toDF方法转换为Dataframe
cate_rating_df = cate_count_df.rdd.map(process_row).toDF(["user_id", "item_id", "rating"])
# 注意：toDF不是每个rdd都有的方法，仅局限于此处的rdd
# 可通过该方法获得 user-cate-matrix
# 但由于cateId字段过多，这里运算量比很大，机器内存要求很高才能执行，否则无法完成任务
# 请谨慎使用

# 但好在我们训练ALS模型时，不需要转换为user-cate-matrix，所以这里可以不用运行
# cate_rating_df.groupBy("userId").povit("cateId").min("rating")
# 用户对类别的偏好打分数据
cate_rating_df

DataFrame[user_id: bigint, item_id: bigint, rating: double]

In [73]:
cate_rating_df.show(10)

+-------+-------+------------------+
|user_id|item_id|            rating|
+-------+-------+------------------+
|4789030|   4457|               0.2|
|2092410|   7468|               0.2|
|5575555|   9760|               0.2|
|9060425|  10359|               0.2|
|6087591|  10592|               0.2|
|1943654|  13172|0.6000000000000001|
| 965227|  13345|               1.4|
|7623095|  17279|               0.2|
|6782916|  18023|               0.2|
| 811445|  18023|               0.2|
+-------+-------+------------------+
only showing top 10 rows



## 2.3.3 推荐列表

In [74]:
from pyspark.ml.recommendation import ALS

# 初始化ALS模型
als = ALS(maxIter=5, regParam=0.01, userCol="user_id", itemCol="item_id", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(cate_rating_df)

In [77]:
userRecs = model.recommendForAllUsers(5)  # 推荐每个用户前5个物品
userRecs.show()

+-------+--------------------+
|user_id|     recommendations|
+-------+--------------------+
|   1238|[[6829, 0.3536468...|
|  39285|[[9130, 2.0197546...|
|  55265|[[2199, 0.1979419...|
|  78064|[[19922, 0.401574...|
|  86082|[[4473, 0.3386553...|
| 102594|[[9130, 1.6270977...|
| 104656|[[15504, 0.501280...|
| 108221|[[10647, 0.628694...|
| 116312|[[25856, 0.524143...|
| 170846|[[8694, 0.7633701...|
| 171642|[[9130, 0.9281934...|
| 173914|[[6829, 0.3536468...|
| 205837|[[3651, 0.3779491...|
| 216014|[[10647, 0.563243...|
| 222730|[[11240, 0.525432...|
| 304448|[[9130, 0.7118179...|
| 365790|[[8694, 0.7879272...|
| 380411|[[10647, 4.859327...|
| 387142|[[10647, 1.481071...|
| 447071|[[10647, 0.563243...|
+-------+--------------------+
only showing top 20 rows



In [78]:
# 将数组中的struct字段展开为列
expanded_df = userRecs.select('user_id', 'recommendations.item_id', 'recommendations.rating')

# 将DataFrame转换为Pandas DataFrame
pandas_df = expanded_df.toPandas()
# 保存为 CSV 文件
pandas_df.to_csv('D:\\Datas\\datas\\User_recommendations.csv', index=False)

## 2.4 评估   <a class="anchor" id="section2_4"></a>

使用 RegressionEvaluator 类来评估模型的均方根误差 (RMSE)。

In [80]:
# 在测试集上进行预测
predictions = model.transform(cate_rating_df)
predictions.show()

+--------+-------+------------------+----------+
| user_id|item_id|            rating|prediction|
+--------+-------+------------------+----------+
| 6459641|  18051|               0.2|0.18755321|
|  435040|  18051|               0.2|0.18755321|
| 8017429|  18051|               0.2|0.18755321|
| 8635765|  18051|               0.2|0.18755321|
| 9183662|  18051|               0.2|0.18755321|
|12340729|  18051|               0.2|0.18755321|
|13502832|  18051|               0.2|0.18755321|
| 6750103|  18051|               0.2|0.18755321|
|11652769|  18051|               0.2|0.18755321|
| 4681255|  18051|               0.2|0.18755321|
|11770172|  18051|               1.0| 0.9377661|
| 1405445|  18051|               0.2|0.18755321|
| 2354849|  18051|               0.2|0.18755321|
| 9041554|  18051|               0.2|0.18755321|
| 1092789|  18051|               0.2|0.18755321|
| 8068318|  18051|               0.2|0.18755321|
|14013955|  18051|               0.2|0.18755321|
|11911447|  18051|0.

In [81]:
from pyspark.ml.evaluation import RegressionEvaluator

# 定义评估器（使用均方根误差和平均绝对误差）
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

# 计算均方根误差
rmse = evaluator_rmse.evaluate(predictions)
print("均方根误差 (RMSE) = {:.4f}".format(rmse))

# 计算平均绝对误差
mae = evaluator_mae.evaluate(predictions)
print("平均绝对误差 (MAE) = {:.4f}".format(mae))

均方根误差 (RMSE) = 0.0356
平均绝对误差 (MAE) = 0.0212


1. RMSE:衡量模型预测值与实际观测值之间差异的指标。RMSE的值越小，表示模型的预测准确性越高。RMSE为0.0356，这意味着模型的预测值平均上下浮动约0.0356个单位。
2. MAE:衡量模型预测值与实际观测值之间绝对差异的指标。MAE的值越小，表示模型的预测准确性越高。MAE为0.0212，这意味着模型的平均预测误差约为0.0212个单位。
3. RMSE为0.0356和MAE为0.0212，可以说模型的预测性能较好，因为较小的值表示模型的预测误差相对较小。

In [82]:
# 关闭Spark会话
spark.stop()

# 3. 优质用户 <a class="anchor" id="section3"></a>

In [22]:
import pandas as pd

# action为大量数据的DataFrame
chunk_size = 100000  # 块大小
chunks = []

# 逐块处理数据
for chunk in pd.read_csv('D:\\Datas\\datas\\filtered_data.csv', chunksize=chunk_size):
    chunks.append(chunk)
action = pd.concat(chunks, axis=0)
action.head(10)

KeyboardInterrupt: 

In [3]:
# 按照用户id分组并统计action数量
# 使用groupby和agg进行分组统计
result = action.groupby(['user_id', 'action']).size().unstack(fill_value=0).reset_index()
result.head()

action,user_id,alipay,cart,click,collect
0,u1,0,0,4,0
1,u10,0,0,7,1
2,u100,0,0,3,0
3,u10000,0,0,9,0
4,u100000,0,0,7,0


In [4]:
result['user_id'] = result['user_id'].str.replace('u', '')

# 显示处理后的DataFrame
result.head()

action,user_id,alipay,cart,click,collect
0,1,0,0,4,0
1,10,0,0,7,1
2,100,0,0,3,0
3,10000,0,0,9,0
4,100000,0,0,7,0


In [5]:
# 按照alipay字段从大到小排序
df_alipay = result.sort_values(by='alipay', ascending=False)
df_alipay.head()

action,user_id,alipay,cart,click,collect
7206421,7704248,151,29,328,0
5060428,4241681,73,65,118,0
2669687,14306949,47,25,273,0
1428402,12303455,38,18,181,24
8062978,9086177,38,2,129,0


In [6]:
#保存方便渲染页面   购买统计
df_alipay.to_csv('D:\\Datas\\datas\\user_action.csv', index=False)

In [ ]:
import pandas as pd

# action为大量数据的DataFrame
chunk_size = 100000  # 块大小
chunks = []

# 逐块处理数据
for chunk in pd.read_csv('D:\\Datas\\datas\\filtered_data.csv', chunksize=chunk_size):
    chunks.append(chunk)
action = pd.concat(chunks, axis=0)
action.head(10)

In [ ]:
# 按照用户id分组并统计action数量
# 使用groupby和agg进行分组统计
result = action.groupby(['user_id', 'action']).size().unstack(fill_value=0).reset_index()
result['user_id'] = result['user_id'].str.replace('u', '')
# 按照alipay字段从大到小排序
df_alipay = result.sort_values(by='alipay', ascending=False)